# SimJEB 2 - training

Trains the surface-stress surrogate on the cached graphs from notebook 1.

**Settings required:** Accelerator **GPU T4**, Internet **ON** (for the pip install).
Add notebook 1's output as an input dataset. Run with *Save & Run All (Commit)*.

### This notebook is meant to be run more than once

At roughly 20-60 s per epoch, a 3,000-epoch budget does not fit a 12-hour session. So
it stops cleanly at a wall-clock limit and writes a full checkpoint every epoch --
model, optimiser, scheduler, history. **Re-running continues from where it stopped.**

To continue: add the *previous run's output* as a second input dataset and set
`RESUME_FROM` below. Early stopping may well fire first.

In [ ]:
GITHUB_REPO = "https://github.com/Vedavamsi-3/simjeb-structural-gnn.git"          # same as notebook 1
DATA        = "/kaggle/input/simjeb-data"   # <- notebook 1's output dataset
RESUME_FROM = ""          # <- a previous run's output, to continue a long job

REPO = "/kaggle/working/simjeb-structural-gnn"

import subprocess, sys, os, shutil
from pathlib import Path

if GITHUB_REPO and not Path(REPO).exists():
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch-geometric", "meshio", "trimesh"], check=True)

sys.path.insert(0, REPO)
os.chdir(REPO)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
# Kaggle mounts input datasets under a name it chooses, so rather than hard-coding
# a path, find whichever mounted dataset actually contains the built graphs.
# Kaggle mounts inputs at a path it chooses, and the depth varies -- sometimes
# /kaggle/input/<name>/graphs, sometimes /kaggle/input/datasets/<name>/graphs. Search
# a few levels rather than assuming one.
_found = [p for p in Path("/kaggle/input").rglob("graphs") if p.is_dir()]
if _found:
    DATA = str(_found[0].parent)
    print("found data at:", DATA)
else:
    print("no graphs/ found under /kaggle/input -- did you Add Data?")
    for p in Path("/kaggle/input").rglob("*"):
        if p.is_dir() and len(p.relative_to("/kaggle/input").parts) <= 2:
            print("   ", p)
_resume = sorted(Path("/kaggle/input").glob("*/outputs/C/checkpoint.pt"))
if _resume and not RESUME_FROM:
    RESUME_FROM = str(_resume[0].parents[2])
    print("found a previous run to resume:", RESUME_FROM)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

OUT = Path("/kaggle/working/outputs")

# Copy a previous run's checkpoint into place -- Kaggle input datasets are read-only,
# and the loop needs to write to the same directory it reads from.
if RESUME_FROM:
    previous = Path(RESUME_FROM) / "outputs" / RUN
    if previous.exists():
        (OUT / RUN).mkdir(parents=True, exist_ok=True)
        for name in ("checkpoint.pt", "best_model.pt", "history.csv"):
            if (previous / name).exists():
                shutil.copy(previous / name, OUT / RUN / name)
        print("resuming from", previous)
    else:
        print("RESUME_FROM set but not found:", previous)

## Configuration

**The run is defined by a file, not by this notebook.** `configs/<RUN>.json` holds every
setting, and the cell below prints them so the log records exactly what ran. Change the
run by changing one word.

Each config carries an `_intent`, a `_changed` block naming what differs from the
previous run and why, and a `_prediction` written before the run — so a result is
informative whichever way it goes.

| run | what it tests |
|---|---|
| `c1_baseline` | The architecture with light regularisation. **Done — test MAE 84.0 MPa against the paper's 60.1 MPa naive baseline, so it lost.** |
| `c2_regularised` | Six fixes at once: stronger weight decay, dropout, Huber loss, no material features, smoothed early stopping, shorter patience. Combined rather than isolated because each run costs ~7 GPU-hours. |

Two settings worth understanding whatever the run:

- **`log_stress`** — peak von Mises reaches 17× the 880 MPa yield at singular corners.
  Without the log, a handful of nodes supply most of the gradient. All metrics invert it
  before scoring.
- **`hidden_dim` / `num_blocks`** — 64 × 8, not the paper's 128 × 15. Depth is set by the
  length scale of stress concentration (8 hops ≈ 7.6 mm, covering fillet radii), not by
  the load path — which is 54 hops away and would need 10.6 GB per graph.

Full reasoning: [`RESULTS.md`](../RESULTS.md).


In [ ]:
from src.train import TrainConfig, train
import json

# The run is defined by a committed file, not by editing this cell. Every run is then
# traceable to an exact config, and changing one never silently redefines another.
RUN = "c2_regularised"

settings = {k: v for k, v in json.load(open(f"configs/{RUN}.json")).items()
            if not k.startswith("_")}
config = TrainConfig(
    graph_dir=f"{DATA}/graphs",
    split_path=f"{DATA}/splits/grouped_split_v1.json",
    out_dir=str(OUT),
    device="cuda", amp=True, num_workers=2, in_memory=True,
    **settings,
)

print(f"run: {config.run_name}")
for key in ("use_material", "use_position", "use_aux_displacement", "huber_delta",
            "hidden_dim", "num_blocks", "dropout", "weight_decay", "patience",
            "val_smoothing", "batch_size"):
    print(f"  {key:22s} {getattr(config, key)}")

for key in ("graph_dir", "split_path"):
    assert Path(getattr(config, key)).exists(), f"{key} not found: {getattr(config, key)}"
print("inputs found")


## Timing check

Run a handful of epochs first and read the real epoch time off the log. That decides
whether 3,000 epochs is one session or four -- and it is much cheaper to learn now
than after ten hours.

In [ ]:
import dataclasses, time

probe = dataclasses.replace(config, run_name="timing_probe", max_epochs=3,
                            patience=10_000, max_hours=0.5)
began = time.time()
probe_result = train(probe)
per_epoch = (time.time() - began) / max(len(probe_result.history), 1)

print(f"\n~{per_epoch:.1f} s per epoch")
print(f"  {config.max_hours} h  -> ~{int(config.max_hours*3600/per_epoch)} epochs per session")
print(f"  {config.max_epochs} epochs -> ~{config.max_epochs*per_epoch/3600:.1f} h total")
shutil.rmtree(OUT / "timing_probe", ignore_errors=True)

## Train

In [ ]:
%%time
result = train(config)

In [ ]:
import pandas as pd
from IPython.display import Image, display

history = pd.read_csv(OUT / RUN / "history.csv")
print(result.stopped_because)
print(f"epochs completed : {len(history)}")
print(f"best epoch       : {result.best_epoch}")
print(f"best val loss    : {result.best_val_loss:.5f}")
print(f"best val R2      : {history.val_r2_mpa.max():.4f}")
print(f"best val MAE     : {history.val_mae_mpa.min():.1f} MPa")
display(Image(str(OUT / RUN / f"loss_curve_{RUN}.png")))

### Reading the curves

- **Validation still falling at the end** -- it stopped on the clock, not on
  convergence. Re-run this notebook with `RESUME_FROM` set to this run's output.
- **Validation flat while training falls** -- overfitting. More weight decay, or a
  smaller model.
- **Both flat and high** -- underfitting. More capacity, or a higher learning rate.
- **Both fallen and levelled together** -- converged. Move to notebook 3.

In [ ]:
final = history.tail(min(50, len(history)))
improving = final.val_loss.iloc[-1] < final.val_loss.iloc[0]
gap = history.val_loss.iloc[-1] / max(history.train_loss.iloc[-1], 1e-12)

print(f"validation still improving over the last {len(final)} epochs: {improving}")
print(f"val/train loss ratio: {gap:.2f}")
if "wall-clock" in result.stopped_because:
    print("\n-> stopped on time, not convergence. Re-run with RESUME_FROM set to this output.")
elif improving:
    print("\n-> early stopping fired but validation was still drifting down; consider more patience.")
else:
    print("\n-> converged. Go to notebook 3.")